# 02. 정답 마스크 검토 (dev + test, 4명 분담)

사진 위에 정답 마스크(빨강)를 겹쳐 보고 버튼을 누른다. 위에서부터 셀을 차례로 실행하면 된다.

| 버튼 | 뜻 |
|---|---|
| **ok** | 마스크가 캐릭터와 맞음 (애매하면 ok) |
| **bad** | 제외할 사진 (공책 줄·엉뚱한 영역을 잡았거나 캐릭터가 없음) |
| **fix** | 고치면 쓸 수 있는 사진 (손발·머리 누락 등) |

- 그림자·복잡한 배경 때문에 **분할이 어려운 사진은 ok**로 둔다 (우리 방법이 풀어야 할 문제다).
- 누를 때마다 `subset/review_<이름>.csv`에 자동 저장된다. 닫았다가 3번 셀을 다시 실행하면 이어서 한다.
- 4명이 `TEAM`을 **같은 순서**로 써야 사진이 겹치거나 빠지지 않는다. 한 사람이 약 125장이다.
- 공유 폴더에 `ad_utils.py`, `subset/`이 있어야 하고 편집 권한이 필요하다 (결과 CSV를 `subset/`에 저장).

In [ ]:
# 1. 설정 — NAME만 자기 이름으로 바꾼다
from google.colab import drive
drive.mount('/content/drive', force_remount=True)   # 바로가기를 새로 만들었어도 보이게 다시 연결
import os, unicodedata

TEAM = ['채연', '서아', '서연', '윤서']       # 4명 이름, 전원이 같은 순서로
NAME = '채연'                                  # 내 이름 (TEAM 중 하나)
FOLDER = '컴퓨터비전'                           # 공유 폴더 이름
BASE = None                                    # 자동으로 못 찾으면 여기에 경로를 직접 적는다


def nfc(s):
    return unicodedata.normalize('NFC', s)


def find_base(name, roots=('/content/drive/MyDrive', '/content/drive/Shareddrives')):
    # 폴더 안에 subset/이 있는 첫 번째 '컴퓨터비전'을 찾는다 (얕은 깊이만 탐색)
    for root in roots:
        for d, dirs, _ in os.walk(root):
            for x in dirs:   # 한글 이름이 다른 방식(NFC/NFD)으로 저장된 경우도 찾는다
                if nfc(x) == nfc(name) and os.path.isdir(os.path.join(d, x, 'subset')):
                    return os.path.join(d, x)
            if d[len(root):].count('/') >= 1:
                dirs[:] = []
    return None


BASE = BASE or find_base(FOLDER)
assert BASE, f"'{FOLDER}' 폴더를 못 찾음: 폴더를 내 드라이브에 바로가기로 추가했는지, 이름이 맞는지 확인"
assert NAME in TEAM, f'NAME이 TEAM에 없음: {NAME}'
print('BASE =', BASE)

In [ ]:
# 2. 검토 도구 (수정하지 않는다)
import os
import cv2
import pandas as pd

STATUSES = ('ok', 'bad', 'fix')


class ReviewState:
    """화면과 무관한 부분: 내 몫 나누기, 기록, 저장, 이어하기."""

    def __init__(self, items, me, n_people, csv_path):
        # items: [(split, im, ann), ...]  (id 순으로 정렬해서 번갈아 나눈다)
        items = sorted(items, key=lambda t: t[1]['id'])
        self.mine = items[me::n_people]
        self.csv_path = csv_path
        self.status = {}
        if os.path.exists(csv_path):
            old = pd.read_csv(csv_path)
            self.status = dict(zip(old.image_id, old.status))
        self.pos = self.first_unlabeled()

    def first_unlabeled(self):
        for i, (_, im, _) in enumerate(self.mine):
            if im['id'] not in self.status:
                return i
        return len(self.mine)

    def label(self, st):
        assert st in STATUSES
        self.status[self.mine[self.pos][1]['id']] = st
        self.save()
        self.pos += 1

    def back(self):
        self.pos = max(0, self.pos - 1)

    def save(self):
        rows = [(sp, im['id'], self.status[im['id']])
                for sp, im, _ in self.mine if im['id'] in self.status]
        pd.DataFrame(rows, columns=['split', 'image_id', 'status']).to_csv(self.csv_path, index=False)

    @property
    def done(self):
        return self.pos >= len(self.mine)


def start(base, me, name, n_people=4):
    import sys
    import ipywidgets as W
    import matplotlib.pyplot as plt
    from IPython.display import display, clear_output
    sys.path.append(base)
    from ad_utils import load_split, load_image, draw_overlay

    sub = os.path.join(base, 'subset')
    items = [(sp, im, ann) for sp in ('dev', 'test') for im, ann in load_split(sub, sp)]
    st = ReviewState(items, me, n_people, os.path.join(sub, f'review_{name}.csv'))

    out = W.Output()
    info = W.HTML()
    btns = {
        'ok':  W.Button(description='ok (맞음)', button_style='success', layout=W.Layout(width='140px', height='44px')),
        'bad': W.Button(description='bad (제외)', button_style='danger', layout=W.Layout(width='140px', height='44px')),
        'fix': W.Button(description='fix (수정필요)', button_style='warning', layout=W.Layout(width='140px', height='44px')),
        'back': W.Button(description='← 이전', layout=W.Layout(width='100px', height='44px')),
    }

    def render():
        n = len(st.mine)
        cnt = {s: sum(v == s for v in st.status.values()) for s in STATUSES}
        with out:
            clear_output(wait=True)
            if st.done:
                info.value = f'<b>완료!</b> {n}장 검토. ok {cnt["ok"]} / bad {cnt["bad"]} / fix {cnt["fix"]}  → {st.csv_path}'
                return
            sp, im, ann = st.mine[st.pos]
            img = load_image(os.path.join(sub, im['file_name']))
            gt = cv2.imread(os.path.join(sub, im['mask_file']), 0) > 127
            fig, axs = plt.subplots(1, 2, figsize=(9, 5))
            axs[0].imshow(img); axs[0].set_title('original'); axs[0].axis('off')
            axs[1].imshow(draw_overlay(img, ann, mask=gt)); axs[1].set_title('GT mask (red)'); axs[1].axis('off')
            plt.tight_layout(); plt.show()
            prev = st.status.get(im['id'], '-')
            info.value = (f'<b>{st.pos + 1} / {n}</b> &nbsp; id {im["id"]} ({sp}) &nbsp; 이전 선택: {prev}'
                          f' &nbsp;|&nbsp; ok {cnt["ok"]} / bad {cnt["bad"]} / fix {cnt["fix"]}')

    def on(key):
        def handler(_):
            if key == 'back':
                st.back()
            elif not st.done:
                st.label(key)
            render()
        return handler

    for k, b in btns.items():
        b.on_click(on(k))
    display(W.VBox([info, W.HBox(list(btns.values())), out]))
    render()
    return st

In [ ]:
# 3. 실행 — 버튼으로 ok / bad / fix 선택
st = start(BASE, me=TEAM.index(NAME), name=NAME, n_people=len(TEAM))

## 4. 합치기 (모두 끝난 뒤 한 명만 실행)
`subset/`의 `review_*.csv` 4개를 하나로 합친다. 장수가 498이고 중복이 없으면 정상이다.

In [ ]:
import glob
import pandas as pd
files = [f for f in glob.glob(f'{BASE}/subset/review_*.csv') if not f.endswith('review_all.csv')]
all_rev = pd.concat([pd.read_csv(f) for f in files])
assert all_rev.image_id.is_unique, '겹치는 사진이 있음 (TEAM 순서가 사람마다 달랐는지 확인)'
print(len(files), '개 파일 /', len(all_rev), '장 /', all_rev.status.value_counts().to_dict())
all_rev.to_csv(f'{BASE}/subset/review_all.csv', index=False)